# 11 · (Optional) Monitoring conditions: degraded footage + your own CCTV images

**Purpose.** Proxy for surveillance / traffic footage: apply low-light, motion-blur, low-resolution and JPEG degradations to a test subset (COCO labels stay valid) and compare detector-only vs the pipeline using the thresholds selected on clean held-out data. A second section runs the online pipeline on a folder of your own images (qualitative).

| | |
|---|---|
| **Proposal objective** | Optional extension (proposal §6.1) |
| **Inputs** | COCO test subset, optional image folder |
| **Outputs** | `results/robustness/*.csv` |
| **Runtime (T4)** | fast profile: ≈2 min per condition (40 images) |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess, shutil  # paths, sys.path, shell commands (git clone, pip), folder copy
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = (not IN_COLAB) and os.path.isdir("/kaggle/working")    # Kaggle notebook (needs Internet ON, see docs/KAGGLE_GUIDE.md)

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"
BRANCH = "kaggle-version"  # git branch to take the latest code from (change it if you push to another branch)

if IN_COLAB:
    from google.colab import drive  # Colab only: mount Google Drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, DRIVE_PROJECT], check=True)
    elif os.path.isdir(os.path.join(DRIVE_PROJECT, ".git")):   # pick up the latest pushed code (never touches your notebooks/results)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "checkout", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], check=False)
    PROJECT_ROOT = DRIVE_PROJECT
elif IN_KAGGLE:
    PROJECT_ROOT = "/kaggle/working/detect-verify-decide"
    if not os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")):
        import glob
        attached = glob.glob("/kaggle/input/*/src/dvd") + glob.glob("/kaggle/input/*/*/src/dvd")   # project attached as a Kaggle Dataset
        if attached:
            shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), PROJECT_ROOT)
        elif REPO_URL:
            url = REPO_URL
            try:   # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets)
                from kaggle_secrets import UserSecretsClient
                url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
            except Exception:
                pass
            r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT_ROOT], capture_output=True)
            if r.returncode:   # (the message deliberately does not echo the command, which could contain the token)
                raise RuntimeError("git clone failed: is Internet ON, REPO_URL / BRANCH right, and for a private repo the GITHUB_TOKEN secret set?")
            subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", REPO_URL])   # never keep the token in .git/config
        else:
            raise RuntimeError("Set REPO_URL (and BRANCH) above, or attach the project as a Kaggle Dataset.")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab / Kaggle already have torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB or IN_KAGGLE:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt  # json files, numpy arrays, pandas tables, matplotlib plots
from dvd.config import load_config, save_selected, results_dir  # load configs/default.yaml, save tuned values, results folders
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache  # seeding, JSON helpers and the resumable JSONL cache
from dvd import paths  # cache-file path helpers

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch  # PyTorch: GPU check and memory statistics
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_annotations, fetch_images, CocoData, load_splits  # COCO annotations, image download, splits
prepare_annotations(cfg)   # annotation json only (the 241 MB zip is cached on Drive)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
fetch_images(cfg, HELDOUT + TEST)   # only these images (a few hundred files), not the whole 778 MB val2017 zip
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.degrade import degrade  # synthetic low-light / blur / low-res / JPEG degradations
from dvd.agents.detector import build_detector  # Detector Agent (Grounding DINO): factory
from dvd.agents.verifier import build_verifier  # Verifier Agent (frozen Qwen2-VL): factory and crop builder
from dvd.pipeline import run_detection_stage, run_verification_stage  # cached offline stages and the online Detect-Verify-Decide pipeline
from dvd.experiments import SplitData, evaluate_row, cached  # methods over identical candidates: sweeps, operating points, evaluation, costs
from dvd.metrics import decision_breakdown  # matching, metrics, sweeps, AUROC, POPE scores
SUB = TEST[: cfg["robustness"]["images"]]
detector = build_detector(cfg, coco.cat_names, coco.name_to_cat_id); verifier = build_verifier(cfg)
iid = SUB[0]
fig, ax = plt.subplots(1, 5, figsize=(22, 4))
for a, c in zip(ax, ["clean"] + cfg["robustness"]["conditions"]):
    a.imshow(coco.image(iid) if c == "clean" else degrade(coco.image(iid), c, iid)); a.set_title(c); a.axis("off")
plt.show()

In [ ]:
sel = cfg["selected"]; rows = []
for cond in ["clean"] + cfg["robustness"]["conditions"]:
    # clean images reuse the test-split caches from notebooks 02/03 (already computed -> no-op)
    split, tag = ("test", "") if cond == "clean" else ("robust", f"_{cond}")
    fn = coco.image if cond == "clean" else (lambda i, c=cond: degrade(coco.image(i), c, i))
    det = run_detection_stage(detector, SUB, paths.det_cache(cfg, split, tag), fn, f"detect {cond}")
    run_verification_stage(verifier, det, SUB, paths.ver_cache(cfg, split, "crop", tag=tag), cfg, fn, view="crop", desc=f"verify {cond}")
    D = SplitData(cfg, coco, split, SUB, det_tag=tag)
    v = cached(paths.ver_cache(cfg, split, "crop", tag=tag), SUB)
    meth = {"kind": "rule", "verdicts": {i: np.asarray(r["verdict"], bool) for i, r in v.items()}}
    base = evaluate_row({"kind": "none"}, D, {"tau": sel["detector_only_tau"]})
    dvd = evaluate_row(meth, D, {"tau_low": sel["tau_low"], "tau_high": sel["tau_high"]})
    b = decision_breakdown(D.labels, {i: D.ev.scores(i) for i in SUB}, dvd.pop("_masks"), sel["tau_low"])
    base.pop("_masks")
    rows.append({"condition": cond, "det_precision": base["precision"], "det_recall": base["recall"], "det_fp_per_img": base["fp_per_image"],
                 "dvd_precision": dvd["precision"], "dvd_recall": dvd["recall"], "dvd_fp_per_img": dvd["fp_per_image"],
                 "fp_reduction": 1 - dvd["fp"] / max(base["fp"], 1), "over_rejection_rate": b["over_rejection_rate"]})
rob = pd.DataFrame(rows); rob.to_csv(results_dir(cfg, "robustness") / "degradations.csv", index=False); rob

## Your own monitoring images (qualitative)
Put `.jpg/.png` frames in `data/custom_images/` on Drive (no labels needed).

In [ ]:
from pathlib import Path  # portable file paths
from PIL import Image  # Pillow image type
from dvd.agents.decision import RuleDecisionAgent  # Decision Agent: the accept/reject rule and the optional LLM aggregator
from dvd.pipeline import DetectVerifyDecide  # cached offline stages and the online Detect-Verify-Decide pipeline
from dvd.viz import show_detections  # plotting helpers (detections overlay, PR frontier, crop gallery)
CUSTOM_DIR = Path(cfg["root"]) / "data" / "custom_images"
files = sorted([p for p in CUSTOM_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:8] if CUSTOM_DIR.exists() else []
print(f"{len(files)} custom images found in {CUSTOM_DIR}")
pipe = DetectVerifyDecide(detector, verifier, RuleDecisionAgent(sel["tau_low"], sel["tau_high"]), cfg)
for p in files:
    img = Image.open(p).convert("RGB"); r = pipe(img)
    keep = [k for k, s in enumerate(r["scores"]) if s >= sel["tau_low"]]
    show_detections(img, [r["boxes"][k] for k in keep], [r["labels"][k] for k in keep], [r["scores"][k] for k in keep],
                    ["accepted" if r["accept"][k] else "rejected" for k in keep], title=f"{p.name} - {r['total_time_s']:.2f}s"); plt.show()